# 05 - Measures Exploration

This notebook calculates key business metrics from the gold layer.

Focus areas:
- Sales, quantity, and order metrics
- Customer and product counts
- Average order value
- Customer purchase coverage
- Business KPI summary report

In [0]:
%sql
/*

Sales Measures Overview


Purpose:
    Calculate core sales metrics including total revenue, quantity sold,
    total orders, and average selling price.

*/

SELECT
    SUM(sales_amount) AS total_sales,
    SUM(quantity) AS total_quantity_sold,
    COUNT(*) AS total_sales_lines,
    COUNT(DISTINCT order_number) AS total_orders,
    ROUND(AVG(price), 2) AS avg_unit_price,
    ROUND(AVG(sales_amount), 2) AS avg_sales_per_line
FROM datawarehouseanalytics_gold.fact_sales;

In [0]:
%sql
/*
Customer and Product Coverage

Purpose:
    Compare the total available customers and products with the customers
    and products that appear in actual sales transactions.
*/

SELECT
    (SELECT COUNT(*) 
     FROM datawarehouseanalytics_gold.dim_customers) AS total_customers,

    (SELECT COUNT(DISTINCT customer_key) 
     FROM datawarehouseanalytics_gold.fact_sales) AS customers_with_orders,

    (SELECT COUNT(*) 
     FROM datawarehouseanalytics_gold.dim_products) AS total_products,

    (SELECT COUNT(DISTINCT product_key) 
     FROM datawarehouseanalytics_gold.fact_sales) AS products_sold;

In [0]:
%sql
/*
Average Order Value

Purpose:
    Calculate order-level revenue first, then measure the average value
    of a customer order.
*/

WITH order_totals AS (
    SELECT
        order_number,
        SUM(sales_amount) AS order_value
    FROM datawarehouseanalytics_gold.fact_sales
    GROUP BY order_number
)

SELECT
    COUNT(*) AS total_orders,
    SUM(order_value) AS total_sales,
    ROUND(AVG(order_value), 2) AS avg_order_value,
    MIN(order_value) AS min_order_value,
    MAX(order_value) AS max_order_value
FROM order_totals;

In [0]:
%sql
/*
Customer Purchase Coverage

Purpose:
    Measure what percentage of customers have placed at least one order.
    This helps understand customer activation in the sales dataset.
*/

SELECT
    COUNT(DISTINCT c.customer_key) AS total_customers,
    COUNT(DISTINCT f.customer_key) AS customers_with_orders,
    ROUND(
        COUNT(DISTINCT f.customer_key) * 100.0 / COUNT(DISTINCT c.customer_key),
        2
    ) AS customer_purchase_coverage_percentage
FROM datawarehouseanalytics_gold.dim_customers c
LEFT JOIN datawarehouseanalytics_gold.fact_sales f
    ON c.customer_key = f.customer_key;

In [0]:
%sql
/*
Business KPI Summary Report

Purpose:
    Combine important business measures into a single report using UNION ALL.
    This provides a quick executive-level overview of the dataset.
*/

WITH order_totals AS (
    SELECT
        order_number,
        SUM(sales_amount) AS order_value
    FROM datawarehouseanalytics_gold.fact_sales
    GROUP BY order_number
)

SELECT 'Total Sales' AS measure_name, CAST(SUM(sales_amount) AS DOUBLE) AS measure_value
FROM datawarehouseanalytics_gold.fact_sales

UNION ALL

SELECT 'Total Quantity Sold', CAST(SUM(quantity) AS DOUBLE)
FROM datawarehouseanalytics_gold.fact_sales

UNION ALL

SELECT 'Total Sales Lines', CAST(COUNT(*) AS DOUBLE)
FROM datawarehouseanalytics_gold.fact_sales

UNION ALL

SELECT 'Total Orders', CAST(COUNT(DISTINCT order_number) AS DOUBLE)
FROM datawarehouseanalytics_gold.fact_sales

UNION ALL

SELECT 'Average Order Value', CAST(ROUND(AVG(order_value), 2) AS DOUBLE)
FROM order_totals

UNION ALL

SELECT 'Average Unit Price', CAST(ROUND(AVG(price), 2) AS DOUBLE)
FROM datawarehouseanalytics_gold.fact_sales

UNION ALL

SELECT 'Total Products', CAST(COUNT(*) AS DOUBLE)
FROM datawarehouseanalytics_gold.dim_products

UNION ALL

SELECT 'Products Sold', CAST(COUNT(DISTINCT product_key) AS DOUBLE)
FROM datawarehouseanalytics_gold.fact_sales

UNION ALL

SELECT 'Total Customers', CAST(COUNT(*) AS DOUBLE)
FROM datawarehouseanalytics_gold.dim_customers

UNION ALL

SELECT 'Customers With Orders', CAST(COUNT(DISTINCT customer_key) AS DOUBLE)
FROM datawarehouseanalytics_gold.fact_sales;